# 01 — Motzkin-Straus and Dirac-3 as a clique sampler

In this notebook we explore the **Motzkin-Straus theorem**, the continuous
quadratic program at the heart of every Dirac-3 pricing call in this
project. We

1. recall the theorem and the "max ½ xᵀA x on the simplex" form,
2. solve the QP **classically** with `scipy.optimize.minimize(method="SLSQP")`
   (multi-start since the problem is non-convex),
3. solve the **same** QP on **Dirac-3** in either replay, cloud, or direct
   mode, and
4. compare what each side recovers as an independent set.

**Motzkin-Straus (1965).** For a simple graph $G$ with adjacency matrix
$A$ and stability number $\alpha(G)$,

$$\frac{1}{2}\left(1 - \frac{1}{\alpha(G)}\right) = \max_{\substack{\mathbf{1}^\top x = 1 \\ x\ge 0}} \tfrac{1}{2}\,x^{\top} A\, x.$$

Maximisers are supported on cliques in $G$. Independent sets in $G$ are
cliques in the complement $\\bar{G}$, so we run Motzkin-Straus on
$\\bar{G}$.

Dirac-3 *minimises* a quadratic with a sum constraint:
$\min\;\tfrac12 x^\top J x \;\text{ s.t. }\; \sum x_i = 1,\; x \ge 0$,
which we instantiate with $J = -A_{\bar{G}}$ — exactly the negative of the
Motzkin-Straus objective. One device call returns many sample vectors;
each sample's *support* is (close to) one clique in $\\bar{G}$ = one
independent set in $G$.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # so that `import _demo_utils` works
                                      # whether run from repo root or notebooks/

import _demo_utils as U
import networkx as nx
import numpy as np
import scipy.optimize
import matplotlib.pyplot as plt


## 1. Pick a small graph

We'll use the **Petersen graph** — 10 vertices, $\alpha = 4$, $\chi = 3$.
It's small enough for an exact SLSQP sweep yet rich enough that the
landscape has multiple local optima.

In [ ]:
G = nx.petersen_graph()
G_bar = nx.complement(G)
A_bar = nx.to_numpy_array(G_bar, nodelist=sorted(G_bar.nodes()))

print(f"Petersen: |V|={G.number_of_nodes()}, |E|={G.number_of_edges()}")
print(f"Complement edges: {G_bar.number_of_edges()}")
print(f"Known α(Petersen) = 4, expected MS optimum = ½(1 - 1/4) = {0.5*(1 - 1/4)}")


## 2. Classical solver — SLSQP with multi-start

The Motzkin-Straus QP is non-convex, so we restart from several Dirichlet
initialisations and keep the best optimum. The constraint $\sum x_i = 1$
is linear and the bounds $x_i \ge 0$ are box bounds; SLSQP handles both.

In [ ]:
def solve_motzkin_straus_slsqp(adj, n_starts=30, seed=0):
    """Maximise ½ xᵀ adj x over the standard simplex via SLSQP multi-start."""
    n = adj.shape[0]
    fun  = lambda x: -0.5 * x @ adj @ x
    jac  = lambda x: -adj @ x
    cons = [{"type": "eq", "fun": lambda x: x.sum() - 1, "jac": lambda x: np.ones(n)}]
    bnds = [(0.0, 1.0)] * n
    rng = np.random.default_rng(seed)
    best, best_history = None, []
    for k in range(n_starts):
        x0 = rng.dirichlet(np.ones(n))
        res = scipy.optimize.minimize(
            fun, x0, jac=jac, method="SLSQP", bounds=bnds, constraints=cons,
            options={"ftol": 1e-10, "maxiter": 200},
        )
        best_history.append(-res.fun)
        if best is None or res.fun < best.fun:
            best = res
    return best, best_history

import time
t0 = time.monotonic()
slsqp_best, slsqp_history = solve_motzkin_straus_slsqp(A_bar, n_starts=30, seed=42)
slsqp_time = time.monotonic() - t0

print(f"SLSQP best objective:    {-slsqp_best.fun:.6f}")
print(f"True MS optimum:         {0.5*(1-1/4):.6f}")
print(f"Multi-start time:        {slsqp_time*1000:.1f} ms")
print(f"Per-start objectives (sorted): "
      f"{sorted(slsqp_history, reverse=True)[:5]} ... {sorted(slsqp_history)[:3]}")


**`★` Insight.** SLSQP converges in milliseconds on this size, but
several restarts land on *suboptimal* local maxima — that's the
non-convexity of Motzkin-Straus. Dirac-3 cope with this by sampling
many restarts in hardware in parallel.

In [ ]:
# Plot the SLSQP optimum as a stem chart
nodes = sorted(G.nodes())
xstar = slsqp_best.x

fig, ax = plt.subplots(figsize=(7, 2.4), constrained_layout=True)
ax.bar(nodes, xstar, color=U.QCI_BLUE, edgecolor="black", linewidth=0.5)
ax.set_xlabel("vertex")
ax.set_ylabel("$x^*_v$")
ax.set_title("SLSQP solution on the simplex — support = max clique in $\\bar{G}$ = MIS in $G$")
ax.set_xticks(nodes)
plt.show()

support = [v for v, p in enumerate(xstar) if p > 1e-3]
print(f"SLSQP support: {support}  (size {len(support)})")
print(f"Is independent in G? {all(not G.has_edge(u, v) for u in support for v in support if u != v)}")


## 3. Dirac-3 sampling — same QP, quantum solver

Pick the backend mode in the next cell:

* **`replay`** — read pre-recorded raw samples from
  `RF-branching/instances/.../raw_samples/` (no API access required).
* **`cloud`** — submit live to the QCi cloud API. You'll be prompted for
  `QCI_TOKEN` if it isn't in your environment or `.env` file.
* **`direct`** — submit to an on-prem Dirac-3 via `eqc-direct`. You'll be
  prompted for `EQC_DIRECT_IP_ADDRESS` (and optional port / cert path).

Note: replay mode reuses bundled samples that were *recorded on a
different graph* (ER(20, 0.7) for the column-generation demo). For
this notebook we still illustrate the Dirac response shape and the
extraction pipeline, but to compare directly against SLSQP on the
**Petersen** complement you need cloud or direct mode.

In [ ]:
# ── Backend toggle ──────────────────────────────────────────────────
BACKEND = "replay"   # "replay" | "cloud" | "direct"

# Direct hardware endpoint (only used when BACKEND == "direct").
# Skill-documented default 172.18.41.79 is offline; 172.18.41.228 is current.
DIRECT_IP = "172.18.41.228"
DIRECT_PORT = 50051
# ────────────────────────────────────────────────────────────────────

# Load .env if present (for QCI_TOKEN convenience)
U.load_dotenv_if_present()

if BACKEND == "replay":
    # The bundled samples are for ER(20, 0.7) — load that PSP so we can
    # demonstrate the Dirac response on a real cached call.
    psp = U.load_psp(1)
    G_dirac = U.psp_to_graph(psp)
    G_dirac_bar = nx.complement(G_dirac.subgraph(
        [v for v in G_dirac.nodes() if psp["subproblem"]["dual_by_label"][str(v)] > 0]
    ))
    layout_dirac = U.psp_to_layout(psp)
    duals_dirac = U.psp_dual_array(psp)
    print(f"Replay graph: ER(20, 0.7) — using bundled call_00000")
else:
    # For cloud / direct we run on the same Petersen graph as SLSQP.
    G_dirac = G
    G_dirac_bar = G_bar
    layout_dirac = nx.kamada_kawai_layout(G_dirac)
    duals_dirac = np.ones(G_dirac.number_of_nodes())   # uniform — first CG iter
    print(f"Live mode {BACKEND}: running on Petersen graph")

oracle = U.make_dirac_oracle(
    BACKEND, method="gibbons", num_samples=100,
    direct_ip_address=DIRECT_IP, direct_port=DIRECT_PORT,
)
print(f"Oracle: {type(oracle).__mro__[1].__name__}, backend={getattr(oracle, 'backend', '?')}")


In [ ]:
# Solve and time
t0 = time.monotonic()
dirac_columns = oracle.solve(G_dirac, duals_dirac)
dirac_time = time.monotonic() - t0

stats = oracle.timer.summary()
print(f"Dirac call: {dirac_time:.2f}s wall ({stats['total_api_seconds']:.2f}s api)")
print(f"Independent sets extracted: {len(dirac_columns)}")
for c in sorted(dirac_columns, key=lambda s: -len(s))[:8]:
    print(f"  IS size {len(c):2d}: {sorted(c)}")


## 4. Side-by-side comparison

* SLSQP gives **one** clique per restart. We did 30 restarts.
* Dirac-3 yields many samples per call, **and** the oracle's extraction
  pipeline tries multiple support thresholds + pruning strategies on each
  sample (multi_prune + randomized_rounding + 1-swap local search), so a
  single call typically lands a handful of distinct cliques.

In [ ]:
# Visualize Dirac-extracted IS over the (replay or live) graph
fig, _ = U.draw_columns_grid(
    G_dirac, list(dirac_columns), layout_dirac,
    ncols=4,
    suptitle=f"Dirac extracted {len(dirac_columns)} unique cliques in $\\bar{{G}}$ "
             f"(= IS in $G$)  —  {BACKEND} mode",
)
plt.show()


In [ ]:
# Quick comparison table
import pandas as pd
slsqp_clique = sorted([v for v, p in enumerate(slsqp_best.x) if p > 1e-3])

rows = [
    {"method": "SLSQP (best of 30)", "objective": f"{-slsqp_best.fun:.4f}",
     "best_clique_size": len(slsqp_clique), "wall_s": f"{slsqp_time:.3f}",
     "restarts/samples": 30},
    {"method": f"Dirac ({BACKEND})", "objective": "—",
     "best_clique_size": max((len(c) for c in dirac_columns), default=0),
     "wall_s": f"{dirac_time:.2f}", "restarts/samples": oracle.num_samples},
]
pd.DataFrame(rows)


## 5. Why use Dirac if SLSQP works?

For tiny graphs SLSQP is unbeatable. The story changes when the QP is
*embedded inside* a larger algorithm — column generation. Each CG
iteration's pricing problem is a **dual-weighted** Motzkin-Straus QP
(Gibbons' weighted formulation), and we want **many** distinct profitable
columns per call to make the master problem progress. Dirac's hardware
parallelism gives us 5–30 unique IS per ~30 s call, where SLSQP would
need that many separate restarts. The next two notebooks (`02a` /
`02b`) show this in action on a real CG run.